# 03. Baseline: CNN Text Classifier on Wine Reviews

Kim-style CNN over frozen GloVe 300d embeddings, on the shared wine splits with
MAX_LEN 100. Same architecture as the SST-2/IMDB/Yelp versions. The training loss
uses pos_weight because wine is ~34% positive. Expect ~5-10 min on a T4
(25,000 training reviews, batch 128).

In [14]:
# ================================================================
# 0. Check the Colab GPU
# ================================================================

# Ask the Colab runtime which GPU we were given.
gpu_info = !nvidia-smi
# Join the command output lines into one printable string.
gpu_info = '\n'.join(gpu_info)
# If the command failed, we are not connected to a GPU runtime.
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.')
else:
    print(gpu_info)

Tue Aug  4 04:33:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   47C    P0            184W /  400W |   12114MiB /  81920MiB |     55%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [15]:
# ================================================================
# 1. Check the Colab RAM
# ================================================================

# psutil reports how much system memory this runtime has.
import psutil
# Convert bytes to gigabytes for a readable number.
ram_gb = psutil.virtual_memory().total / 1e9
# Print the available RAM so we know which runtime type we received.
print('Your runtime has {:.1f} gigabytes of available RAM'.format(ram_gb))

Your runtime has 179.4 gigabytes of available RAM


In [16]:
# ================================================================
# 2. Basic imports and reproducibility
# ================================================================

# pathlib gives clean, operating-system-safe file paths.
from pathlib import Path

# random controls Python-level randomness.
import random

# re is used for simple text cleaning and tokenization.
import re

# Counter helps build the vocabulary from token frequencies.
from collections import Counter

# numpy handles numeric arrays outside PyTorch.
import numpy as np

# pandas handles CSV data tables.
import pandas as pd

# torch is the deep learning framework used across all AMIC notebooks.
import torch

# nn contains PyTorch neural-network modules.
from torch import nn

# F contains functional operations such as relu, softmax, and softplus.
import torch.nn.functional as F

# Dataset and DataLoader create mini-batches for training.
from torch.utils.data import Dataset, DataLoader

# Metrics for classification quality and calibration.
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss, log_loss

# matplotlib draws simple training curves.
import matplotlib.pyplot as plt

# This seed keeps runs as reproducible as possible; it matches the wine BAMIC notebooks.
SEED = 20260526

# Seed Python's random module.
random.seed(SEED)

# Seed NumPy's random generator.
np.random.seed(SEED)

# Seed PyTorch's CPU generator.
torch.manual_seed(SEED)

# Seed all CUDA devices if a GPU is available.
torch.cuda.manual_seed_all(SEED)

# Pick the Colab GPU when available; otherwise fall back to CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the selected device so we know what this notebook is using.
print('Using device:', DEVICE)

Using device: cuda


In [17]:
# ================================================================
# 3. Mount Google Drive and define project paths
# ================================================================

# Mount Google Drive so Colab can read and write project files.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

# This is the same Google Drive project folder used by the wine and SST-2 notebooks.
PROJECT_DIR = Path('/content/drive/MyDrive/AMIC project')

# All wine benchmark files live inside this subfolder.
BENCH_DIR = PROJECT_DIR / 'wine_benchmark'

# The shared, prepared wine splits are stored here by notebook 00.
DATA_DIR = BENCH_DIR / 'data'

# This notebook writes all of its results into its own output folder.
EXPERIMENT_NAME = 'wine_cnn_glove'
OUTPUT_DIR = BENCH_DIR / 'outputs' / EXPERIMENT_NAME

# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Print the paths so we can verify them before loading data.
print('DATA_DIR  :', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
DATA_DIR  : /content/drive/MyDrive/AMIC project/wine_benchmark/data
OUTPUT_DIR: /content/drive/MyDrive/AMIC project/wine_benchmark/outputs/wine_cnn_glove


In [18]:
# ================================================================
# 4. Load the shared wine splits prepared by notebook 00
# ================================================================

# Stop early with a clear message if notebook 00 has not been run yet.
for name in ['wine_train.csv', 'wine_valid.csv', 'wine_test.csv']:
    if not (DATA_DIR / name).exists():
        raise FileNotFoundError(f'Missing {DATA_DIR / name}. Run 00_prepare_wine_data.ipynb first.')

# Read the three prepared splits from Drive (the train file is ~5 MB, quick).
train_df = pd.read_csv(DATA_DIR / 'wine_train.csv')
valid_df = pd.read_csv(DATA_DIR / 'wine_valid.csv')
test_df = pd.read_csv(DATA_DIR / 'wine_test.csv')

# Force the text column to string in case pandas parsed something unusually.
for df in [train_df, valid_df, test_df]:
    df['text'] = df['text'].astype(str)

# Print shapes and label balance. NOTE: unlike the other benchmarks, wine is NOT
# balanced - about 34% of reviews score 90+ - so expect a positive rate near 0.34.
print('train:', train_df.shape, '| positive rate:', round(train_df['y'].mean(), 4))
print('valid:', valid_df.shape, '| positive rate:', round(valid_df['y'].mean(), 4))
print('test :', test_df.shape, '| positive rate:', round(test_df['y'].mean(), 4))

# Show one example review (truncated for display).
print(train_df['text'].iloc[0][:300], '...')

train: (25000, 3) | positive rate: 0.342
valid: (3000, 3) | positive rate: 0.342
test : (10000, 3) | positive rate: 0.342
Offers a diffuse, medium-bodied mix of cherry cola and pie, with a touch of chicory, turning simpler on the finish. Drink now. 625 cases made. ...


In [19]:
# ================================================================
# 5. Tokenization, vocabulary, and fixed-length encoding
# ================================================================

# This regular expression keeps word-like tokens, numbers, and contractions
# such as "it's" as single tokens (same tokenizer as the wine and SST-2 notebooks).
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")


def clean_text(text):
    """Convert raw text to a normalized lowercase string."""
    # Guard against missing values.
    if not isinstance(text, str):
        text = ''
    # Lowercase so 'Good' and 'good' share one embedding.
    text = text.lower()
    # Collapse repeated whitespace into single spaces.
    text = re.sub(r'\s+', ' ', text).strip()
    # Return the normalized string.
    return text


def tokenize(text):
    """Tokenize text into a list of lowercase word-like tokens."""
    return TOKEN_RE.findall(clean_text(text))


# Wine reviews are SHORT (mean ~32 words, maximum 100 in this corpus), so the
# 100-token window from the original AMIC work covers every review completely.
MAX_LEN = 100

# MIN_FREQ = 2 matches the original wine AMIC/BAMIC notebooks, so the vocabulary
# (and therefore the published wine numbers) stays comparable.
MIN_FREQ = 2

# Count token frequencies over the TRAINING split only (never valid/test, to avoid leakage).
token_counts = Counter()
for text in train_df['text']:
    token_counts.update(tokenize(text))

# Index 0 is reserved for padding; index 1 is reserved for unknown words.
word_to_id = {'<PAD>': 0, '<UNK>': 1}

# Add every frequent-enough token to the vocabulary.
for word, count in token_counts.most_common():
    if count >= MIN_FREQ:
        word_to_id[word] = len(word_to_id)

# Build the reverse map for printing words back from ids.
id_to_word = {idx: word for word, idx in word_to_id.items()}

# Report the vocabulary size and the most common tokens as a sanity check.
print('Vocabulary size including PAD and UNK:', len(word_to_id))
print('Most common tokens:', token_counts.most_common(10))


def encode_text(text):
    """Convert one review into a fixed-length list of token ids."""
    # Map each token to its id, or to UNK when the token is out of vocabulary.
    ids = [word_to_id.get(tok, 1) for tok in tokenize(text)]
    # Truncate reviews that are longer than MAX_LEN.
    ids = ids[:MAX_LEN]
    # Remember the real (unpadded) length.
    length = len(ids)
    # Pad short reviews with zeros up to MAX_LEN.
    ids = ids + [0] * (MAX_LEN - len(ids))
    # Return both the ids and the real length.
    return ids, length


def encode_dataframe(df):
    """Encode a whole dataframe into id matrices, lengths, and label arrays."""
    encoded = [encode_text(t) for t in df['text']]
    x = np.array([e[0] for e in encoded], dtype=np.int64)
    lengths = np.array([e[1] for e in encoded], dtype=np.int64)
    y = df['y'].to_numpy().astype(np.float32)
    return x, lengths, y


# Encode all three splits (38k short reviews - a few seconds).
x_train, len_train, y_train = encode_dataframe(train_df)
x_valid, len_valid, y_valid = encode_dataframe(valid_df)
x_test, len_test, y_test = encode_dataframe(test_df)

# Report how much text the 256-token window truncates.
print('Share of train reviews truncated at MAX_LEN:', round((len_train == MAX_LEN).mean(), 3))

# Print the encoded shapes to confirm everything lines up.
print('x_train:', x_train.shape, 'y_train:', y_train.shape)
print('x_valid:', x_valid.shape, 'y_valid:', y_valid.shape)
print('x_test :', x_test.shape, 'y_test :', y_test.shape)

Vocabulary size including PAD and UNK: 5210
Most common tokens: [('and', 57996), ('with', 24692), ('cases', 23842), ('a', 23808), ('the', 21999), ('now', 21213), ('drink', 20745), ('made', 18466), ('of', 17692), ('through', 17657)]
Share of train reviews truncated at MAX_LEN: 0.0
x_train: (25000, 100) y_train: (25000,)
x_valid: (3000, 100) y_valid: (3000,)
x_test : (10000, 100) y_test : (10000,)


In [20]:
# ================================================================
# 6. Load GloVe 300d embeddings for our vocabulary
# ================================================================

# GloVe 6B vectors are 300-dimensional, matching the wine BAMIC setup.
EMBED_DIM = 300

# The GloVe text file lives in the shared word_embedding folder of the Drive project.
GLOVE_PATH = PROJECT_DIR / 'word_embedding' / 'glove.6B.300d.txt'

# Fail early with a clear message if the embedding file is missing.
if not GLOVE_PATH.exists():
    raise FileNotFoundError(f'GloVe file not found: {GLOVE_PATH}. '
                            'Upload glove.6B.300d.txt to the word_embedding folder in Drive.')


def build_random_embedding_matrix(vocab_size, embedding_dim):
    """Create a small random embedding matrix with a zero padding row."""
    # A fixed-seed generator keeps the random rows reproducible.
    rng = np.random.default_rng(SEED)
    # Small random values for words that GloVe does not cover.
    matrix = rng.normal(loc=0.0, scale=0.05, size=(vocab_size, embedding_dim)).astype(np.float32)
    # Row 0 is the PAD token and must stay all-zero.
    matrix[0] = 0.0
    # Return the initialized matrix.
    return matrix


def load_glove_for_vocab(glove_path, word_to_id, embedding_dim=300):
    """Load only the GloVe vectors whose words appear in our vocabulary."""
    # Start from the random matrix so uncovered words keep small random vectors.
    matrix = build_random_embedding_matrix(len(word_to_id), embedding_dim)
    # Track how many vocabulary words GloVe covers.
    found = 0
    # A set makes the membership test fast inside the file loop.
    needed_words = set(word_to_id.keys())
    # Stream the large GloVe file line by line to stay memory-friendly.
    with open(glove_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            # Each line is: word value_1 ... value_300.
            parts = line.rstrip().split(' ')
            # Skip malformed lines defensively.
            if len(parts) != embedding_dim + 1:
                continue
            # First field is the word itself.
            word = parts[0]
            # Ignore words that are not in our vocabulary.
            if word not in needed_words:
                continue
            # Copy this word's vector into its row of the matrix.
            matrix[word_to_id[word]] = np.asarray(parts[1:], dtype=np.float32)
            # Count the successful match.
            found += 1
    # Keep the PAD row exactly zero after loading.
    matrix[0] = 0.0
    # Report the coverage so we can compare with the wine corpus (92.7% there).
    print(f'Loaded {found:,} / {len(word_to_id):,} vocabulary vectors from GloVe '
          f'({found / max(1, len(word_to_id)):.1%}).')
    # Return the matrix as a torch tensor ready for nn.Embedding.
    return torch.tensor(matrix, dtype=torch.float32)


# Build the embedding matrix for this vocabulary.
embedding_matrix = load_glove_for_vocab(GLOVE_PATH, word_to_id, EMBED_DIM)

# Confirm the final shape: (vocabulary size, embedding dimension).
print('embedding_matrix:', tuple(embedding_matrix.shape))

Loaded 4,988 / 5,210 vocabulary vectors from GloVe (95.7%).
embedding_matrix: (5210, 300)


In [21]:
# ================================================================
# 7. Dataset and DataLoader
# ================================================================

class WineTextDataset(Dataset):
    """Dataset wrapper that serves token ids, real lengths, and labels."""

    def __init__(self, x, lengths, y):
        # Store token ids as long tensors (required by nn.Embedding).
        self.x = torch.tensor(x, dtype=torch.long)
        # Store real review lengths for models that need them.
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        # Store labels as float tensors (required by BCEWithLogitsLoss).
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        # The dataset length is the number of reviews.
        return self.x.size(0)

    def __getitem__(self, idx):
        # Return one example as a dictionary of tensors.
        return {'input_ids': self.x[idx], 'length': self.lengths[idx], 'label': self.y[idx]}


# Wine reviews are short (100 tokens max), so a large batch is comfortable on a T4.
BATCH_SIZE = 128

# Build the three split datasets.
train_ds = WineTextDataset(x_train, len_train, y_train)
valid_ds = WineTextDataset(x_valid, len_valid, y_valid)
test_ds = WineTextDataset(x_test, len_test, y_test)

# Build the mini-batch loaders; only training data is shuffled.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Print the number of batches per split as a sanity check.
print('train batches:', len(train_loader))
print('valid batches:', len(valid_loader))
print('test batches :', len(test_loader))

train batches: 196
valid batches: 24
test batches : 79


In [22]:
# ================================================================
# 8. CNN model definition
# ================================================================

class TextCNN(nn.Module):
    """Kim-style CNN: parallel convolutions over word embeddings + max pooling."""

    def __init__(self, embedding_matrix, filter_sizes=(3, 4, 5), n_filters=100, dropout=0.5):
        # Initialize the parent nn.Module.
        super().__init__()
        # Load the pretrained GloVe matrix; freeze=True keeps embeddings fixed like BAMIC.
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # The embedding dimension comes from the matrix itself.
        embed_dim = embedding_matrix.shape[1]
        # One Conv1d per filter width; each looks at n-grams of that width.
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=n_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        # Dropout regularizes the pooled feature vector.
        self.dropout = nn.Dropout(dropout)
        # The final linear layer maps pooled features to one sentiment logit.
        self.fc = nn.Linear(n_filters * len(filter_sizes), 1)

    def forward(self, input_ids):
        # Look up embeddings: [batch, seq_len, embed_dim].
        x = self.embedding(input_ids)
        # Conv1d expects channels first: [batch, embed_dim, seq_len].
        x = x.transpose(1, 2)
        # Apply each convolution followed by ReLU.
        conv_outputs = [F.relu(conv(x)) for conv in self.convs]
        # Global max pooling over time keeps the strongest n-gram signal per filter.
        pooled = [out.max(dim=2).values for out in conv_outputs]
        # Concatenate the pooled features from all filter widths.
        features = torch.cat(pooled, dim=1)
        # Apply dropout before the final classifier.
        features = self.dropout(features)
        # Produce one logit per review; squeeze removes the trailing dimension.
        return self.fc(features).squeeze(-1)


# Training hyperparameters for this baseline.
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 6

# Build the model and move it to the GPU.
model = TextCNN(embedding_matrix).to(DEVICE)

# Print the parameter count to keep model sizes comparable across notebooks.
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Trainable parameters: 360601


In [23]:
# ================================================================
# Metric helpers (identical to the wine BAMIC notebook, for fair comparison)
# ================================================================

def binary_metrics_from_probs(probs, labels, threshold=0.5):
    """Compute common binary metrics from predicted positive-class probabilities."""
    # Turn probabilities into hard 0/1 predictions at the given threshold.
    pred = (probs >= threshold).astype(int)
    # Fraction of documents classified correctly.
    acc = accuracy_score(labels, pred)
    # Harmonic mean of precision and recall for the positive class.
    f1 = f1_score(labels, pred, zero_division=0)
    # Threshold-free ranking quality; needs both classes present.
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) == 2 else np.nan
    # Mean squared error between probabilities and true labels.
    brier = brier_score_loss(labels, probs)
    # Negative log-likelihood of the true labels under the predicted probabilities.
    nll = log_loss(labels, np.clip(probs, 1e-7, 1 - 1e-7), labels=[0, 1])
    # Return everything in one dictionary.
    return {'acc': acc, 'f1': f1, 'auc': auc, 'brier': brier, 'nll': nll}


def expected_calibration_error(probs, labels, n_bins=10):
    """Compute a simple expected calibration error (ECE)."""
    # Create equally spaced probability bins between 0 and 1.
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    # Accumulate the weighted calibration gap here.
    ece = 0.0
    # Loop over each bin edge pair.
    for lo, hi in zip(bins[:-1], bins[1:]):
        # Include the right edge for the final bin.
        if hi == 1.0:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        # Skip bins that contain no predictions.
        if not np.any(mask):
            continue
        # Average predicted probability inside this bin.
        conf = probs[mask].mean()
        # Empirical positive rate inside this bin.
        acc = labels[mask].mean()
        # Weight the |confidence - accuracy| gap by the bin frequency.
        ece += np.abs(conf - acc) * mask.mean()
    # Return the scalar ECE value.
    return float(ece)

In [24]:
# ================================================================
# 10. Training loop
# ================================================================

# Class weighting compensates for any label imbalance (SST-2 is roughly balanced,
# so this weight will be close to 1, but we keep the same recipe as the wine notebooks).
n_pos = float(y_train.sum())
n_neg = float(len(y_train) - y_train.sum())
pos_weight = torch.tensor([n_neg / max(1.0, n_pos)], dtype=torch.float32, device=DEVICE)

# Logits-based binary cross-entropy is numerically stable.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# AdamW is the standard optimizer used across all our notebooks.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Keep one history row per epoch for the history.csv output.
history = []


@torch.no_grad()
def predict_probs(loader):
    """Return predicted positive-class probabilities for every example in a loader."""
    # Switch off dropout for deterministic evaluation.
    model.eval()
    # Collect per-batch probability arrays here.
    all_probs = []
    # Loop over mini-batches without building gradients.
    for batch in loader:
        # Move token ids to the GPU.
        input_ids = batch['input_ids'].to(DEVICE)
        # Forward pass returns one logit per sentence.
        logits = model(input_ids)
        # Sigmoid converts logits to probabilities; move them back to CPU.
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
    # Concatenate all batches into one flat array.
    return np.concatenate(all_probs)


# Loop over training epochs.
for epoch in range(1, EPOCHS + 1):
    # Put the model in training mode (enables dropout).
    model.train()
    # Track the running loss of this epoch.
    epoch_losses = []
    # Loop over training mini-batches.
    for batch in train_loader:
        # Move inputs and labels to the GPU.
        input_ids = batch['input_ids'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        # Clear old gradients before the new backward pass.
        optimizer.zero_grad(set_to_none=True)
        # Forward pass: one logit per sentence.
        logits = model(input_ids)
        # Compute the weighted binary cross-entropy loss.
        loss = criterion(logits, labels)
        # Backward pass: compute gradients.
        loss.backward()
        # Clip gradients so one bad batch cannot destabilize training.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        # Update the model parameters.
        optimizer.step()
        # Remember this batch's loss value.
        epoch_losses.append(float(loss.detach().cpu()))

    # Evaluate on the validation split at the end of every epoch.
    valid_probs_epoch = predict_probs(valid_loader)
    # Compute the standard metric set on validation predictions.
    valid_metrics = binary_metrics_from_probs(valid_probs_epoch, y_valid)
    # Compute the validation calibration error.
    valid_ece = expected_calibration_error(valid_probs_epoch, y_valid)
    # Store one row of history for this epoch.
    history.append({'epoch': epoch, 'train_loss': float(np.mean(epoch_losses)),
                    'val_acc': valid_metrics['acc'], 'val_f1': valid_metrics['f1'],
                    'val_auc': valid_metrics['auc'], 'val_ece': valid_ece})
    # Print a one-line progress summary.
    print(f"Epoch {epoch:02d} | loss={np.mean(epoch_losses):.4f} | "
          f"val_acc={valid_metrics['acc']:.4f} | val_auc={valid_metrics['auc']:.4f} | "
          f"val_ece={valid_ece:.4f}")

# Convert the history to a dataframe and save it for later comparison plots.
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'history.csv', index=False)

# Show the full training history table.
history_df

Epoch 01 | loss=0.5398 | val_acc=0.8447 | val_auc=0.9364 | val_ece=0.0808
Epoch 02 | loss=0.4263 | val_acc=0.8650 | val_auc=0.9423 | val_ece=0.0355
Epoch 03 | loss=0.3863 | val_acc=0.8607 | val_auc=0.9425 | val_ece=0.0665
Epoch 04 | loss=0.3608 | val_acc=0.8617 | val_auc=0.9448 | val_ece=0.0601
Epoch 05 | loss=0.3277 | val_acc=0.8703 | val_auc=0.9455 | val_ece=0.0398
Epoch 06 | loss=0.3033 | val_acc=0.8627 | val_auc=0.9452 | val_ece=0.0574


,epoch,train_loss,val_acc,val_f1,val_auc,val_ece
0,1,0.539845,0.844667,0.793623,0.936411,0.080824
1,2,0.426286,0.865000,0.809052,0.942310,0.035549
2,3,0.386315,0.860667,0.810517,0.942544,0.066501
3,4,0.360760,0.861667,0.811962,0.944798,0.060052
4,5,0.327747,0.870333,0.818986,0.945524,0.039831
5,6,0.303340,0.862667,0.812897,0.945227,0.057433


In [25]:
# ================================================================
# Final predictions on all three splits
# ================================================================

# IMPORTANT: train_loader was built with shuffle=True, which re-shuffles the data on
# every pass. Using it for evaluation would return probabilities in a random order that
# no longer lines up with y_train, making the train metrics look like coin-flipping.
# We therefore build a NON-shuffled copy of the training data just for evaluation.
train_eval_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Predict probabilities for every split with the trained model.
train_probs = predict_probs(train_eval_loader)
valid_probs = predict_probs(valid_loader)
test_probs = predict_probs(test_loader)

# Rename the label arrays to the names used by the shared saving cell.
y_train_true = y_train.astype(int)
y_valid_true = y_valid.astype(int)
y_test_true = y_test.astype(int)

# Quick sanity check on the shapes.
print('test_probs:', test_probs.shape, '| first 5:', np.round(test_probs[:5], 3))

test_probs: (10000,) | first 5: [0.491 0.471 0.996 0.999 0.924]


In [26]:
# ================================================================
# Save final metrics and predictions to Drive
# ================================================================

# Collect one metrics row per split so the comparison notebook can read them later.
final_metric_rows = []

# Loop over the three splits with their predicted probabilities and true labels.
for split_name, probs, labels in [('train', train_probs, y_train_true),
                                  ('valid', valid_probs, y_valid_true),
                                  ('test', test_probs, y_test_true)]:
    # Compute accuracy, F1, AUC, Brier, and NLL for this split.
    metrics = binary_metrics_from_probs(probs, labels)
    # Compute the calibration error for this split.
    ece = expected_calibration_error(probs, labels)
    # Store everything in one row.
    final_metric_rows.append({'split': split_name, **metrics, 'ece': ece})
    # Print the row so we can see the result immediately.
    print(split_name, {k: round(v, 4) for k, v in metrics.items()}, 'ece=', round(ece, 4))

# Convert the rows into a small dataframe.
final_metrics_df = pd.DataFrame(final_metric_rows)

# Save the metrics table into this notebook's output folder.
final_metrics_df.to_csv(OUTPUT_DIR / 'final_metrics.csv', index=False)

# Also save the raw test predictions for later error analysis.
pd.DataFrame({'y_true': y_test_true, 'p_positive': test_probs}).to_csv(
    OUTPUT_DIR / 'test_predictions.csv', index=False)

# Confirm where everything was written.
print('Saved final_metrics.csv and test_predictions.csv to:', OUTPUT_DIR)

train {'acc': 0.9325, 'f1': 0.908, 'auc': np.float64(0.988), 'brier': np.float64(0.0519), 'nll': 0.1792} ece= 0.0659
valid {'acc': 0.8627, 'f1': 0.8129, 'auc': np.float64(0.9452), 'brier': np.float64(0.0971), 'nll': 0.3065} ece= 0.0574
test {'acc': 0.8622, 'f1': 0.813, 'auc': np.float64(0.946), 'brier': np.float64(0.097), 'nll': 0.3069} ece= 0.0605
Saved final_metrics.csv and test_predictions.csv to: /content/drive/MyDrive/AMIC project/wine_benchmark/outputs/wine_cnn_glove
